In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/boulesnane/best-fold5-val/best_fold5_val.json
/kaggle/input/models/boulesnane/yolo-disease/other/default/1/Yolo_disease.pt
/kaggle/input/models/boulesnane/bestyoloteethfliplr0/other/default/1/best.pt


In [3]:
!pip install huggingface_hub
from huggingface_hub import login
import shutil

from huggingface_hub import snapshot_download

HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    login(token=HF_TOKEN)
path = snapshot_download(
    repo_id="ibrahimhamamci/DENTEX",
    repo_type="dataset"
)

print(path)

print("DENTEX ready")

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

/root/.cache/huggingface/hub/datasets--ibrahimhamamci--DENTEX/snapshots/7b27ccc8e342dcb774f69adc6ca5e6c09fefce93
DENTEX ready


In [4]:
import zipfile
import os

base_path = "/root/.cache/huggingface/hub/datasets--ibrahimhamamci--DENTEX/snapshots/7b27ccc8e342dcb774f69adc6ca5e6c09fefce93/DENTEX"

for file in ["training_data.zip", "validation_data.zip", "test_data.zip"]:
    zip_path = os.path.join(base_path, file)
    extract_path = os.path.join(base_path, file.replace(".zip", ""))

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

    print(f"Extracted {file}")

Extracted training_data.zip
Extracted validation_data.zip
Extracted test_data.zip


In [6]:
!pip install ultralytics==8.4.47 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.0 MB/s eta 0:00:0000:0100:01


In [7]:
# #############################################################################
#  ABLATION of the disease-tooth association score (Eq. fusion-score)
#      s(d,t) = a * IoU(d,t) + (1-a) * exp(-lam * dist(d,t))
#  Paper setting: a = 0.70, lam = 10  (normalized xywh, as in generate_structured_json)
#
#  Test data: all DENTEX diagnosis images (705 train + 50 val) = 3,711 lesions,
#  each with its GT FDI number. The Stage-1 tooth model never saw these images.
#  For every lesion box we pick the tooth with the highest score and check
#  whether its FDI number equals the DENTEX FDI number.
#
#  CELL A1 : run Stage-1 once on all images (cache)          ~3-4 min
#  CELL A2 : grid over a and lam + baselines (pure python)    seconds
#  CELL A3 : paper table (LaTeX) for the chosen rows
#  CELL A4 : (optional) end-to-end with PREDICTED disease boxes (fold-5 val only)
# #############################################################################

# %% CELL A1 — Stage-1 teeth on every diagnosis image (same settings as the pipeline)
import os, json, math
import numpy as np
from collections import Counter, defaultdict
from ultralytics import YOLO

TOOTH_WEIGHTS = "/kaggle/input/models/boulesnane/bestyoloteethfliplr0/other/default/1/best.pt"
TOOTH_CONF    = 0.25          # as in generate_structured_json
AGNOSTIC      = True          # new pipeline setting

DX = ("/root/.cache/huggingface/hub/datasets--ibrahimhamamci--DENTEX/snapshots/"
      "7b27ccc8e342dcb774f69adc6ca5e6c09fefce93/DENTEX")
def first_existing(*cands):
    for c in cands:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(cands[0])

TR_DIR = first_existing(f"{DX}/training_data/training_data/quadrant-enumeration-disease",
                        f"{DX}/training_data/quadrant-enumeration-disease")
VA_DIR = first_existing(f"{DX}/validation_data/validation_data/quadrant_enumeration_disease",
                        f"{DX}/validation_data/quadrant_enumeration_disease")
SOURCES = [(f"{TR_DIR}/train_quadrant_enumeration_disease.json", f"{TR_DIR}/xrays"),
           (f"{DX}/validation_triple.json",                      f"{VA_DIR}/xrays")]
DISEASE = {0: "Impacted", 1: "Caries", 2: "Periapical", 3: "Deep Caries"}
FDI_LIST = [q * 10 + t for q in range(1, 5) for t in range(1, 9)]

# lesions: (path, [xc,yc,w,h] normalized, gt_fdi, disease)
lesions = []
for js, img_dir in SOURCES:
    d = json.load(open(js))
    info = {im["id"]: im for im in d["images"]}
    for a in d["annotations"]:
        im = info[a["image_id"]]; W, H = im["width"], im["height"]
        x, y, w, h = a["bbox"]
        lesions.append((os.path.join(img_dir, im["file_name"]),
                        [(x + w / 2) / W, (y + h / 2) / H, w / W, h / H],
                        (a["category_id_1"] + 1) * 10 + a["category_id_2"] + 1,
                        DISEASE[a["category_id_3"]]))
paths = sorted({l[0] for l in lesions if os.path.exists(l[0])})

tooth_model = YOLO(TOOTH_WEIGHTS)
TEETH = {}                     # path -> list of (fdi, [xc,yc,w,h] normalized)
for p in paths:
    r = tooth_model.predict(p, conf=TOOTH_CONF, agnostic_nms=AGNOSTIC, verbose=False)[0]
    best = {}
    for c, cf, b in zip(r.boxes.cls.tolist(), r.boxes.conf.tolist(), r.boxes.xywhn.tolist()):
        fdi = FDI_LIST[int(c)]
        if fdi not in best or cf > best[fdi][0]:          # same de-duplication as the pipeline
            best[fdi] = (cf, b)
    TEETH[p] = [(fdi, b) for fdi, (cf, b) in best.items()]
lesions = [l for l in lesions if l[0] in TEETH]
print(f"{len(lesions)} lesions in {len(paths)} images | "
      f"{np.mean([len(v) for v in TEETH.values()]):.1f} teeth per image")


# %% CELL A2 — grid search + baselines
def xyxy(b):
    xc, yc, w, h = b
    return xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2

def iou(a, b):
    a, b = xyxy(a), xyxy(b)
    iw = max(0, min(a[2], b[2]) - max(a[0], b[0])); ih = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter + 1e-12)

def ioa(d, t):                 # share of the DISEASE box inside the tooth box
    a, b = xyxy(d), xyxy(t)
    iw = max(0, min(a[2], b[2]) - max(a[0], b[0])); ih = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    return iw * ih / ((a[2]-a[0])*(a[3]-a[1]) + 1e-12)

def dist(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])

# pre-compute IoU / IoA / distance for every lesion-tooth pair once
PAIRS = []                     # per lesion: (gt_fdi, disease, fdis[], iou[], ioa[], dist[])
for p, db, g, dis in lesions:
    T = TEETH[p]
    PAIRS.append((g, dis, [f for f, _ in T],
                  np.array([iou(db, b) for _, b in T]),
                  np.array([ioa(db, b) for _, b in T]),
                  np.array([dist(db, b) for _, b in T])))

MIRROR = {1: 2, 2: 1, 3: 4, 4: 3}
def status(g, p):
    if p is None: return "no tooth"
    if p == g: return "correct"
    if p == MIRROR[g // 10] * 10 + g % 10: return "side swapped"
    if p // 10 == g // 10 and abs(p % 10 - g % 10) == 1: return "off by one"
    return "other wrong"

def evaluate(score_fn):
    allc, disc = Counter(), defaultdict(Counter)
    for g, dis, fdis, I, A, D in PAIRS:
        if not fdis:
            st = "no tooth"
        else:
            s = score_fn(I, A, D)
            st = status(g, fdis[int(np.argmax(s))])
        allc[st] += 1; disc[dis][st] += 1
    n = sum(allc.values())
    return 100 * allc["correct"] / n, allc, disc

ALPHAS  = [0.0, 0.3, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
LAMBDAS = [1, 5, 10, 20, 50]
grid = {}
for a in ALPHAS:
    for lam in LAMBDAS:
        grid[(a, lam)] = evaluate(lambda I, A, D, a=a, lam=lam: a * I + (1 - a) * np.exp(-lam * D))[0]

print("Correct FDI assignment (%) — rows: IoU weight a, columns: distance decay lam")
print(f"{'a / lam':>8}" + "".join(f"{l:>8}" for l in LAMBDAS))
for a in ALPHAS:
    print(f"{a:>8.1f}" + "".join(f"{grid[(a, l)]:>7.2f}{'*' if (a, l) == (0.7, 10) else ' '}" for l in LAMBDAS))
best = max(grid, key=grid.get)
print(f"\n* = paper setting (a=0.70, lam=10): {grid[(0.7, 10)]:.2f}%   |   best in grid: a={best[0]}, lam={best[1]} -> {grid[best]:.2f}%")

BASELINES = {
    "IoU only (a = 1)":                 lambda I, A, D: I + 1e-9 * np.exp(-D),   # tiny tie-break by distance
    "IoU only, no tie-break":           lambda I, A, D: I,
    "Nearest centre only (a = 0)":      lambda I, A, D: -D,
    "Max IoA (lesion inside tooth)":    lambda I, A, D: A + 1e-9 * np.exp(-D),
    "Ours (a = 0.70, lam = 10)":        lambda I, A, D: 0.7 * I + 0.3 * np.exp(-10 * D),
}
print(f"\n{'Association rule':<34}{'correct':>9}{'off by 1':>10}{'side':>7}{'other':>7}" + "".join(f"{d:>12}" for d in DISEASE.values()))
RES = {}
for name, fn in BASELINES.items():
    acc, c, dc = evaluate(fn); RES[name] = (acc, c, dc); n = sum(c.values())
    print(f"{name:<34}{acc:8.1f}%{100*c['off by one']/n:9.1f}%{100*c['side swapped']/n:6.1f}%{100*c['other wrong']/n:6.1f}%"
          + "".join(f"{100*dc[d]['correct']/max(sum(dc[d].values()),1):11.1f}%" for d in DISEASE.values()))

# how many lesions have NO overlapping tooth at all (where the distance term is the only signal)
no_ov = sum(1 for g, dis, fdis, I, A, D in PAIRS if len(I) and I.max() == 0)
print(f"\nLesions with zero IoU to every tooth: {no_ov} ({100*no_ov/len(PAIRS):.1f}%) "
      f"-> by disease: {dict(Counter(dis for g, dis, fdis, I, A, D in PAIRS if len(I) and I.max() == 0))}")


# %% CELL A3 — LaTeX table
rows = [("IoU only ($\\alpha=1$)", "IoU only (a = 1)"),
        ("Centre distance only ($\\alpha=0$)", "Nearest centre only (a = 0)"),
        ("Intersection over lesion area (IoA)", "Max IoA (lesion inside tooth)"),
        ("Composite score, Eq.~(\\ref{eq:fusion-score}) ($\\alpha=0.7$, $\\lambda=10$)", "Ours (a = 0.70, lam = 10)")]
print("\\begin{tabular}{lccccc}\n\\toprule")
print("\\textbf{Association rule} & \\textbf{All} & \\textbf{Imp.} & \\textbf{Caries} & \\textbf{Periap.} & \\textbf{Deep car.} \\\\\n\\midrule")
for tex, key in rows:
    acc, c, dc = RES[key]
    print(f"{tex} & {acc:.1f} & " + " & ".join(f"{100*dc[d]['correct']/max(sum(dc[d].values()),1):.1f}" for d in DISEASE.values()) + " \\\\")
print("\\bottomrule\n\\end{tabular}")
print("\nGrid (a x lam) for the text / supplementary:")
for a in ALPHAS:
    print(f"$\\alpha={a:.1f}$ & " + " & ".join(f"{grid[(a, l)]:.1f}" for l in LAMBDAS) + " \\\\")


# %% CELL A4 — (optional) end-to-end: PREDICTED disease boxes on fold-5 val images only
# The disease model must be the fold-5 model, and only fold-5 val images are used (never seen by it).
# Each predicted box is matched to a DENTEX lesion of the same class (IoU >= 0.5);
# then we check the FDI number given by each association rule.
RUN_END_TO_END = False
DISEASE_WEIGHTS = "/kaggle/input/models/boulesnane/yolo-disease/other/default/1/Yolo_disease.pt"   # fold-5 model
FOLD5_VAL_JSON  = "/kaggle/input/datasets/boulesnane/best-fold5-val/best_fold5_val.json"  # as in the VLM notebook

if RUN_END_TO_END:
    dm = YOLO(DISEASE_WEIGHTS)
    print("disease model trained on:", dm.ckpt["train_args"]["data"])
    val_names = {it["image_info"]["file_name"] for it in json.load(open(FOLD5_VAL_JSON))}
    by_path = defaultdict(list)
    for l in lesions:
        if os.path.basename(l[0]) in val_names:
            by_path[l[0]].append(l)
    E2E = []
    for p, L in by_path.items():
        r = dm.predict(p, imgsz=1024, conf=0.25, verbose=False)[0]
        for c, b in zip(r.boxes.cls.int().tolist(), r.boxes.xywhn.tolist()):
            cand = [(iou(b, l[1]), l) for l in L if l[3] == DISEASE[c]]
            if not cand or max(cand)[0] < 0.5:
                continue                                    # false positive: no GT FDI to compare with
            g = max(cand, key=lambda z: z[0])[1][2]
            T = TEETH[p]
            E2E.append((g, DISEASE[c], [f for f, _ in T],
                        np.array([iou(b, t) for _, t in T]), np.array([ioa(b, t) for _, t in T]),
                        np.array([dist(b, t) for _, t in T])))
    PAIRS_BAK, PAIRS = PAIRS, E2E
    print(f"\nEnd-to-end: {len(E2E)} correctly detected lesions in {len(by_path)} fold-5 val images")
    for name, fn in BASELINES.items():
        print(f"  {name:<34}{evaluate(fn)[0]:6.1f}%")
    PAIRS = PAIRS_BAK

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
3711 lesions in 724 images | 28.3 teeth per image
Correct FDI assignment (%) — rows: IoU weight a, columns: distance decay lam
 a / lam       1       5      10      20      50
     0.0  94.26   94.26   94.26   94.26   94.26 
     0.3  94.56   94.48   94.48   94.45   94.45 
     0.5  94.56   94.53   94.50   94.50   94.56 
     0.6  94.56   94.56   94.53   94.53   94.56 
     0.7  94.56   94.56   94.56*  94.56   94.56 
     0.8  94.56   94.56   94.56   94.56   94.56 
     0.9  94.56   94.56   94.56   94.56   94.56 
     1.0  94.56   94.56   94.56   94.56   94.56 

* = paper setting (a=0.70, lam=10): 94.56%   |   best in grid: a=0.3, lam=1 -> 94.56%

Association rule                 